# Corrected User MoE Experiments - 8 Configuration Comparison

This notebook supersedes the original `48.4%` comparison. The original notebook split the complete table before rolling evaluation, so each expert built test windows anchored to a different last job. Its global and MoE metrics therefore covered different jobs and were not comparable.

This version uses current `main` and tests the same factorial design correctly:

1. **User routing:** pooled users vs power-user experts
2. **Wallclock routing:** one model vs data-derived requested-wallclock experts
3. **Time weighting:** flat vs exponential decay at rate `0.05`

All configurations use one shared rolling grid, current submission-time feature policy, matched XGBoost hyperparameters, and fallback predictions for sparse experts. The runner refuses to compare experiments unless the split timestamps, exact test rows, scored-row count, and true targets are identical.

**Agreed Kestrel methodology:** 120 six-hour windows, 60-day training lookback, benchmark window 2025-03-29 through 2025-06-26.

## 1. Setup

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root")


REPO_ROOT = find_repo_root()
RUNNER = REPO_ROOT / "docs" / "benchmarking" / "run_user_moe_experiments.py"
CHECKPOINT = REPO_ROOT / "workspace" / "user_moe_experiments_corrected.json"
LOG_PATH = REPO_ROOT / "workspace" / "user_moe_experiments_corrected.log"
DATA_PATH = REPO_ROOT / "workspace" / "data" / "datasets" / "nlr_kestrel" / "data.parquet"

print(f"Repository: {REPO_ROOT}")
print(f"Data exists: {DATA_PATH.exists()}")
print(f"Checkpoint exists: {CHECKPOINT.exists()}")

## 2. Run or Resume

The long experiment runs in a Python script with progressive logging and a checkpoint after each completed configuration. Set `RUN_EXPERIMENTS=True` to launch it from this notebook, or run the printed command in a terminal. On interruption, rerun with `--resume`.

In [ ]:
RUN_EXPERIMENTS = False

command = [
    sys.executable,
    str(RUNNER),
    "--resume",
    "--data-path",
    str(DATA_PATH),
    "--checkpoint",
    str(CHECKPOINT),
    "--log-path",
    str(LOG_PATH),
    "--n-windows",
    "120",
    "--test-window-hours",
    "6",
    "--training-lookback-days",
    "60",
    "--max-svd-components",
    "256",
    "--target-max-one-hot-width",
    "2048",
    "--n-estimators",
    "200",
    "--max-depth",
    "8",
    "--learning-rate",
    "0.05",
    "--estimator-n-jobs",
    "12",
    "--quiet-model",
]

print(" ".join(command))
if RUN_EXPERIMENTS:
    subprocess.run(command, cwd=REPO_ROOT, check=True)

## 3. Load Checkpoint

In [ ]:
if not CHECKPOINT.exists():
    raise FileNotFoundError("Run or resume the experiment first; checkpoint does not exist")

state = json.loads(CHECKPOINT.read_text())
print(f"Status: {state['status']}")
print(f"Completed: {len(state['results'])}/8 configurations")
print(f"Git commit: {state['git_commit']}")
print(f"Slice rows: {state['data']['slice_rows']:,}")

## 4. Feature-Policy Audit

This cell verifies that post-hoc fields were present only as ignored columns, never as model features.

In [ ]:
policy = state["feature_policy"]
print("Eligible submission-time features:")
print("  " + ", ".join(policy["eligible"]))
print("\nPost-hoc/identifier fields present but ignored:")
print("  " + (", ".join(policy["forbidden_present_but_ignored"]) or "(none present)"))

forbidden = {
    "job_id",
    "job_state",
    "exit_code",
    "allocated_cpus",
    "num_cores_alloc",
    "num_nodes_alloc",
    "start_time",
    "end_time",
    "runtime_seconds",
}
assert forbidden.isdisjoint(policy["eligible"])

## 5. Same-Row Validity Checks

Every configuration must have the same row count and the same cryptographic fingerprints for test-row indices and true targets.

In [ ]:
results = list(state["results"].values())
if len(results) != 8:
    print("Run is incomplete; checks apply to completed configurations only.")

row_counts = {result["rows_scored"] for result in results}
test_hashes = {result["test_rows_sha256"] for result in results}
target_hashes = {result["y_true_sha256"] for result in results}

assert len(row_counts) == 1, row_counts
assert len(test_hashes) == 1, test_hashes
assert len(target_hashes) == 1, target_hashes

print(f"Identical scored rows: {next(iter(row_counts)):,}")
print(f"Test-row fingerprint: {next(iter(test_hashes))}")
print(f"Target fingerprint:   {next(iter(target_hashes))}")

## 6. Results

In [ ]:
df = pd.DataFrame(results).sort_values("key").reset_index(drop=True)
baseline_mae = float(df.loc[df["key"] == "1", "mae"].iloc[0])
baseline_rmse = float(df.loc[df["key"] == "1", "rmse"].iloc[0])
baseline_median = float(df.loc[df["key"] == "1", "median_absolute_error"].iloc[0])

df["mae_change_pct"] = (df["mae"] / baseline_mae - 1.0) * 100.0
df["rmse_change_pct"] = (df["rmse"] / baseline_rmse - 1.0) * 100.0
df["median_change_pct"] = (df["median_absolute_error"] / baseline_median - 1.0) * 100.0

display_columns = [
    "key",
    "label",
    "mae",
    "mae_change_pct",
    "rmse",
    "rmse_change_pct",
    "median_absolute_error",
    "median_change_pct",
    "underprediction_ratio",
    "rows_scored",
    "elapsed_minutes",
]
df[display_columns].style.format(
    {
        "mae": "{:,.1f}s",
        "mae_change_pct": "{:+.1f}%",
        "rmse": "{:,.1f}s",
        "rmse_change_pct": "{:+.1f}%",
        "median_absolute_error": "{:,.1f}s",
        "median_change_pct": "{:+.1f}%",
        "underprediction_ratio": "{:.1f}%",
        "rows_scored": "{:,}",
        "elapsed_minutes": "{:.1f}",
    }
)

## 7. Factor Effects

These are paired comparisons because each pair differs by exactly one factor and scores identical jobs. Negative percentages are improvements.

In [ ]:
by_key = {row["key"]: row for row in results}


def paired_change(first, second, metric="mae"):
    return (by_key[second][metric] / by_key[first][metric] - 1.0) * 100.0


effects = pd.DataFrame(
    [
        {
            "factor": "Wallclock routing",
            "context": "flat, pooled users",
            "change_pct": paired_change("1", "2"),
        },
        {
            "factor": "Wallclock routing",
            "context": "decay, pooled users",
            "change_pct": paired_change("3", "4"),
        },
        {
            "factor": "Wallclock routing",
            "context": "flat, user routing",
            "change_pct": paired_change("5", "6"),
        },
        {
            "factor": "Wallclock routing",
            "context": "decay, user routing",
            "change_pct": paired_change("7", "8"),
        },
        {
            "factor": "User routing",
            "context": "flat, single wallclock",
            "change_pct": paired_change("1", "5"),
        },
        {
            "factor": "User routing",
            "context": "decay, single wallclock",
            "change_pct": paired_change("3", "7"),
        },
        {
            "factor": "User routing",
            "context": "flat, wallclock routing",
            "change_pct": paired_change("2", "6"),
        },
        {
            "factor": "User routing",
            "context": "decay, wallclock routing",
            "change_pct": paired_change("4", "8"),
        },
        {
            "factor": "Time decay",
            "context": "pooled, single wallclock",
            "change_pct": paired_change("1", "3"),
        },
        {
            "factor": "Time decay",
            "context": "pooled, wallclock routing",
            "change_pct": paired_change("2", "4"),
        },
        {
            "factor": "Time decay",
            "context": "user routing, single wallclock",
            "change_pct": paired_change("5", "7"),
        },
        {
            "factor": "Time decay",
            "context": "user + wallclock routing",
            "change_pct": paired_change("6", "8"),
        },
    ]
)
effects.style.format({"change_pct": "{:+.2f}%"})

## 8. Error and Cost Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
labels = df["label"].tolist()
positions = np.arange(len(df))
colors = ["#4c78a8" if not user else "#59a14f" for user in df["user_routing"]]

for axis, metric, title in [
    (axes[0], "mae", "Mean Absolute Error"),
    (axes[1], "rmse", "Root Mean Squared Error"),
    (axes[2], "median_absolute_error", "Median Absolute Error"),
]:
    axis.barh(positions, df[metric], color=colors)
    axis.set_title(title)
    axis.set_yticks(positions)
    axis.set_yticklabels(labels if axis is axes[0] else [])
    axis.invert_yaxis()
    axis.set_xlabel("seconds (lower is better)")

fig.suptitle("Corrected 8-Configuration MoE Ablation - Identical Test Jobs")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(labels, df["elapsed_minutes"], color=colors)
ax.set_ylabel("runtime (minutes)")
ax.set_title("Evaluation Cost")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 9. Routing Diagnostics

In [ ]:
routing_rows = []
for result in results:
    routing = result.get("routing")
    if routing is None:
        continue
    routing_rows.append(
        {
            "key": result["key"],
            "label": result["label"],
            "experts_per_window_mean": routing["experts_per_window_mean"],
            "routed_row_fraction": routing["routed_row_fraction"],
            "fallback_rows": routing.get("fallback_rows", 0),
            "power_users_last": routing["power_users_last"],
            "bin_edges_hours_last": routing["bin_edges_hours_last"],
        }
    )

routing_df = pd.DataFrame(routing_rows)
routing_df.style.format(
    {
        "experts_per_window_mean": "{:.1f}",
        "routed_row_fraction": "{:.1%}",
        "fallback_rows": "{:,}",
    }
)

## Interpretation Rules

- Do not reuse the original `48.4%` figure.
- Treat MAE, RMSE, and median absolute error as separate outcomes; an improvement in one does not imply improvement in all.
- Report routing cost and fallback share alongside predictive accuracy.
- These Kestrel results characterize one system and one agreed time window; cross-system behavior must be measured rather than assumed.
- A decay result is valid only for the stated lookback and rate. The 120-day decay study should remain a separate experiment from this 60-day routing ablation.